# 01. Descripción y Calidad del Dataset GHAW

**Objetivo:** Cargar el dataset relacional Parquet generado por Miner, evaluar su integridad y calidad de datos, resolver inconsistencias y guardar las tablas limpias preparadas en `eda/data/processed/`

In [ ]:
import os
import json
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Crear directorio de datos procesados si no existe
os.makedirs("data/processed", exist_ok=True)

# Definir rutas de los datos
RAW_DIR = "../dataset_parquet"
PROCESSED_DIR = "data/processed"

## 1. Origen y Carga de los Datos

El dataset analiza flujos de trabajo basados en agentes extraídos de GitHub. El dataset original se encuentra alojado en Hugging Face Hub:
* **Enlace Hugging Face**: `(https://huggingface.co/datasets/ry-02/github-agentic-workflows-dataset`

### Estructura de las Tablas:
1. **`repositories`**: Representa los repositorios de GitHub analizados. Cada fila equivale a **un repositorio único**.
2. **`workflow_files`**: Almacena el contenido en texto plano de los workflows `.md`. Cada fila representa **un archivo Markdown único**.
3. **`workflow_metadata`**: Contiene los atributos extraídos del Frontmatter YAML. Cada fila representa **los metadatos estructurados de un archivo Markdown**.

In [ ]:
df_repos = pd.read_parquet(os.path.join(RAW_DIR, "repositories.parquet"))
df_files = pd.read_parquet(os.path.join(RAW_DIR, "workflow_files.parquet"))
df_meta = pd.read_parquet(os.path.join(RAW_DIR, "workflow_metadata.parquet"))

print("¡Tablas Parquet cargadas exitosamente!")

## 2. Descripción de las Tablas y Sus Relaciones

In [ ]:
def print_table_summary(df, name, pk, fk=None):
    print(f"=== TABLA: {name} ===")
    print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
    print(f"Clave Primaria (PK): {pk}")
    print(f"Clave Foránea (FK): {fk}")
    print("\nTipos de Datos y Columnas:")
    print(df.dtypes)
    print("-" * 50)

print_table_summary(df_repos, "repositories", pk="repository_id")
print_table_summary(df_files, "workflow_files", pk="file_id", fk="repository_id")
print_table_summary(df_meta, "workflow_metadata", pk="metadata_id", fk="file_id")

# Contadores únicos
unique_repos = df_repos["repository_id"].nunique()
unique_files = df_files["file_id"].nunique()

summary_counts = pd.DataFrame({
    "Entidad": ["Repositorios Únicos", "Archivos Markdown Únicos"],
    "Conteo": [unique_repos, unique_files]
})
display(summary_counts)

## 3. Revisión de Calidad
Evaluamos los siguientes 5 aspectos de calidad sobre el dataset:
1. Valores ausentes por columna.
2. Filas totalmente duplicadas.
3. Claves primarias (PK) duplicadas o vacías.
4. Claves foráneas (FK) huérfanas (sin correspondencia en la tabla padre).
5. Inconsistencias en tipos de datos.

In [ ]:
# 1. Valores Nulos
nulls_df = pd.DataFrame({
    "repositories_nulls": df_repos.isnull().sum(),
    "workflow_files_nulls": df_files.isnull().sum(),
    "workflow_metadata_nulls": df_meta.isnull().sum()
}).fillna("-")

print("--- 1. Valores Ausentes por Columna ---")
display(nulls_df)

# 2. Filas Duplicadas
duplicates_summary = pd.DataFrame({
    "Tabla": ["repositories", "workflow_files", "workflow_metadata"],
    "Filas Duplicadas": [
        df_repos.duplicated().sum(),
        df_files.duplicated().sum(),
        df_meta.duplicated().sum()
    ]
})
print("--- 2. Filas Duplicadas ---")
display(duplicates_summary)

# 3. Integridad de Claves Primarias (PK)
pk_summary = pd.DataFrame({
    "Tabla": ["repositories", "workflow_files", "workflow_metadata"],
    "PK": ["repository_id", "file_id", "metadata_id"],
    "PKs Nulas": [
        df_repos["repository_id"].isnull().sum(),
        df_files["file_id"].isnull().sum(),
        df_meta["metadata_id"].isnull().sum()
    ],
    "PKs Repetidas": [
        df_repos["repository_id"].duplicated().sum(),
        df_files["file_id"].duplicated().sum(),
        df_meta["metadata_id"].duplicated().sum()
    ]
})
print("--- 3. Integridad de Claves Primarias ---")
display(pk_summary)

# 4. Integridad Referencial (FK Huérfanas)
orphaned_files = df_files[~df_files["repository_id"].isin(df_repos["repository_id"])]
orphaned_meta = df_meta[~df_meta["file_id"].isin(df_files["file_id"])]

fk_summary = pd.DataFrame({
    "Relación FK": ["workflow_files -> repositories", "workflow_metadata -> workflow_files"],
    "Registros Huérfanos": [len(orphaned_files), len(orphaned_meta)]
})
print("--- 4. Claves Foráneas sin Correspondencia ---")
display(fk_summary)